# Notebook 1 — ETL: Extracción, Transformación y Carga
**Proyecto:** Academic Performance, Course Load, and Student Persistence  
**Institución:** Instituto Tecnológico Metropolitano (ITM)  
**Programa:** Desarrollo de Software  
**Semestres cubiertos:** 2022-1 a 2025-1

---
## Objetivo
Leer los 7 archivos Excel originales (uno por semestre), aplicar un proceso
unificado de limpieza y estandarización, y exportar un archivo limpio por
semestre listo para el análisis longitudinal.

## Flujo
```
Original_Data/Data/*.xlsx
        │
        ▼  limpiar_parametros_liquidacion()
        │  cleaning_function()
        ▼
ETL _process/Cleaning_Data/*_ETLclean.xlsx
```

## 1. Librerías y configuración de rutas
Se utilizan rutas relativas basadas en la ubicación del proyecto para que el
notebook funcione en cualquier entorno (local, servidor, etc.) sin modificar
las rutas manualmente.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path

# ── Rutas ────────────────────────────────────────────────────────────────────
# El notebook vive en:  Proyecto_Final_STEM/ETL _process/
# La raíz del proyecto: Proyecto_Final_STEM/
NOTEBOOK_DIR   = Path().resolve()                          # ETL _process/
PROJECT_ROOT   = NOTEBOOK_DIR.parent                       # Proyecto_Final_STEM/

CARPETA_ENTRADA = PROJECT_ROOT / 'Original_Data' / 'Data'
CARPETA_SALIDA  = NOTEBOOK_DIR / 'Cleaning_Data'

print(f'Entrada : {CARPETA_ENTRADA}')
print(f'Salida  : {CARPETA_SALIDA}')
assert CARPETA_ENTRADA.exists(), f'No se encontró la carpeta de entrada: {CARPETA_ENTRADA}'


## 2. Exploración inicial de archivos
Se inspeccionan las hojas y columnas de un archivo de muestra para entender
la estructura de los datos antes de aplicar las transformaciones.

In [ ]:
archivos = sorted(CARPETA_ENTRADA.glob('*.xlsx'))
if not archivos:
    raise FileNotFoundError(f'No se encontraron archivos .xlsx en {CARPETA_ENTRADA}')

archivo_ejemplo = archivos[0]
print(f'Archivos encontrados ({len(archivos)}):')
for a in archivos:
    print(f'  • {a.name}')

print(f'\nInspeccionando: {archivo_ejemplo.name}')
xls   = pd.ExcelFile(archivo_ejemplo)
hojas = xls.sheet_names

print(f'\nHojas encontradas ({len(hojas)}):')
for h in hojas:
    print(f'  - {h}')

print('\nColumnas por hoja:')
for hoja in hojas:
    try:
        df = pd.read_excel(archivo_ejemplo, sheet_name=hoja, nrows=0)
        print(f'\n  [{hoja}]  →  {list(df.columns)}')
    except Exception as e:
        print(f'  ⚠ No se pudo leer "{hoja}": {e}')


## 3. Funciones de transformación
### 3.1 `limpiar_parametros_liquidacion`
Extrae de la hoja `Parametros Liquidación` dos variables clave:
`Caracter Colegio` (oficial / privado) y `Estrato` socioeconómico.
La hoja tiene un formato semi-estructurado que requiere parseo manual.

In [ ]:
def limpiar_parametros_liquidacion(path: Path) -> pd.DataFrame:
    """
    Transforma la hoja 'Parametros Liquidación' en un DataFrame tabular con
    columnas: Carne, Apellidos y Nombres, Documento, Caracter Colegio, Estrato.

    La hoja mezcla encabezados de sección (p. ej. 'Estrato 1 Caracter OFICIAL')
    con filas de datos; esta función los separa y asigna el estrato/caracter
    correcto a cada estudiante.
    """
    df = pd.read_excel(path, sheet_name='Parametros Liquidación', header=None, dtype=str)

    datos          = []
    estrato_actual = None
    caracter_actual= None

    for _, fila in df.iterrows():
        fila = fila.dropna().tolist()
        if not fila:
            continue

        texto = str(fila[0]).strip()

        # Detectar líneas de encabezado (p. ej. 'Estrato 1 Caracter OFICIAL')
        if 'Estrato' in texto and 'Caracter' in texto:
            m = re.search(r'Estrato\s+(\d+).*Caracter\s+(\w+)', texto, re.IGNORECASE)
            if m:
                estrato_actual  = m.group(1)
                caracter_actual = m.group(2).upper()
            continue

        # Ignorar la fila de cabecera de columnas
        if 'carné' in texto.lower():
            continue

        # Guardar registro de estudiante
        if estrato_actual and caracter_actual and len(fila) >= 3:
            carne, nombre, documento = fila[:3]
            datos.append([carne, nombre, documento, caracter_actual, estrato_actual])

    return pd.DataFrame(
        datos,
        columns=['Carne', 'Apellidos y Nombres', 'Documento', 'Caracter Colegio', 'Estrato']
    )


### 3.2 Funciones auxiliares de notas
Los porcentajes de nota están en formato `'30%-3.5'` (peso-nota).  
`separar_valor` parsea ese string, y `reagrupar_fila` los homogeniza en
5 columnas estándar de 20 % cada una (`nota20_1` … `nota20_5`).

In [ ]:
def separar_valor(x) -> tuple:
    """
    Parsea una celda con formato 'PESO%-NOTA' (e.g. '30%-3.5').

    Returns
    -------
    (peso_decimal, nota_float)  — (0, 0) si el valor es inválido o nulo.
    """
    try:
        if pd.isna(x):
            return 0, 0
        peso_str, nota_str = str(x).split('-')
        peso = float(peso_str.replace('%', '')) / 100
        nota = float(nota_str)
        return peso, nota
    except Exception:
        return 0, 0


def reagrupar_fila(fila: pd.Series) -> pd.Series:
    """
    Convierte todas las columnas de nota de una fila en 5 notas estándar
    de 20 % de peso (nota20_1 … nota20_5).

    Estrategia:
    - Las notas que ya pesan exactamente 20 % se conservan directamente.
    - Las notas con peso < 20 % se agrupan: su aporte ponderado se divide
      en bloques de 20 %, generando notas equivalentes.
    - Se rellena con 0 hasta completar 5 columnas.
    """
    pesos_notas = [separar_valor(fila[col]) for col in fila.index]
    pesos_notas = [(p, n) for p, n in pesos_notas if p > 0]

    notas_20    = [n for p, n in pesos_notas if abs(p - 0.2) < 1e-6]
    otros       = [(p, n) for p, n in pesos_notas if p < 0.2]

    aporte_total = sum(p * n for p, n in otros)
    peso_total   = sum(p for p, _ in otros)

    notas_extra = []
    if peso_total > 0:
        nota_eq    = aporte_total / peso_total
        n_bloques  = round(peso_total / 0.2)
        notas_extra = [nota_eq] * n_bloques

    notas_finales = notas_20 + notas_extra
    while len(notas_finales) < 5:
        notas_finales.append(0)

    return pd.Series(notas_finales[:5], index=[f'nota20_{i}' for i in range(1, 6)])


### 3.3 `cleaning_function` — Limpieza y unificación de hojas
Esta función consolida todas las hojas relevantes de un archivo semestral:
- Unifica hojas de notas de las 4 materias fundacionales.
- Elimina columnas personales (teléfono, carnet, N°).
- Integra variables sociodemográficas (estrato, edad, barrio, campus).
- Crea indicadores binarios: `Desplazado` y `Discapacidad`.
- Estandariza notas parciales a 5 columnas de 20 % (`nota20_1` … `nota20_5`).

In [ ]:
# Mapeo de nombres de hoja de Excel → nombre de materia estandarizado
MAPA_HOJAS_MATERIAS = {
    'XRCD03'         : 'Calculo Diferencial',
    'Notas XRCD03'   : 'Calculo Diferencial',
    'Notas XRAL03'   : 'Algebra Lineal',
    'Notas 580304002': '580304002',
    'Notas 000506001': '000506001',
}


def cleaning_function(path: Path, hoja_param_liq: pd.DataFrame) -> pd.DataFrame:
    """
    Limpia y consolida todas las hojas de un archivo semestral.

    Parameters
    ----------
    path            : ruta al archivo .xlsx original
    hoja_param_liq  : DataFrame ya limpio de 'Parametros Liquidación'

    Returns
    -------
    DataFrame consolidado con: variables sociodemográficas + notas
    estandarizadas + indicadores binarios Desplazado / Discapacidad.
    """
    data    = pd.ExcelFile(path)
    df_all  = {sheet: data.parse(sheet) for sheet in data.sheet_names}
    df_all['Parametros Liquidación'] = hoja_param_liq

    # ── Consolidar hojas de notas ─────────────────────────────────────────
    lista = []
    for hoja, materia in MAPA_HOJAS_MATERIAS.items():
        if hoja in df_all:
            df_hoja = df_all[hoja].copy()
            df_hoja['Materia'] = materia
            lista.append(df_hoja)

    notas = pd.concat(lista, ignore_index=True)
    notas = notas.drop(columns=['N°', 'Teléfono', 'Carnet'], errors='ignore')
    notas = notas.sort_values('Apellidos y Nombres').reset_index(drop=True)
    notas['Documento'] = notas['Documento'].astype(int)

    # ── Variables sociodemográficas ───────────────────────────────────────
    promedio = df_all['Promedio Calificaciones'].copy()
    promedio['Documento'] = promedio['Documento'].astype(int)
    promedio['Estrato']   = promedio['Estrato'].astype(int)
    promedio = promedio[['Documento', 'Promedio', 'Barrio', 'Estrato',
                          'Comuna', 'Campus', 'Promedio Acumulado', 'Estado']]

    edad = df_all['Edad'].copy()
    edad['Documento'] = edad['Documento'].astype(int)
    edad['Edad']      = edad['Edad'].astype(int)
    edad = edad[['Documento', 'Edad']]

    liquidacion = df_all['Parametros Liquidación'].copy()
    liquidacion['Documento'] = liquidacion['Documento'].astype(int)
    liquidacion = liquidacion[['Documento', 'Caracter Colegio']]

    # ── Unión por Documento ───────────────────────────────────────────────
    completo = promedio.merge(notas,        on='Documento', how='inner')
    completo = edad.merge(completo,          on='Documento', how='inner')
    completo = liquidacion.merge(completo,   on='Documento', how='inner')

    # ── Indicadores binarios ──────────────────────────────────────────────
    desplazados  = df_all['Desplazados'].copy()
    desplazados['Documento'] = desplazados['Documento'].astype(int)
    completo['Desplazado'] = completo['Documento'].isin(desplazados['Documento']).astype(int)

    discapacidad = df_all['Discapacidad'].copy()
    discapacidad['Documento'] = discapacidad['Documento'].astype(int)
    completo['Discapacidad'] = completo['Documento'].isin(discapacidad['Documento']).astype(int)

    # ── Estandarizar notas → nota20_1 … nota20_5 ─────────────────────────
    cols_nota = [c for c in completo.columns if 'Nota' in c]
    notas_std = completo[cols_nota].apply(reagrupar_fila, axis=1)
    completo  = completo.drop(columns=cols_nota)
    completo  = pd.concat([completo, notas_std], axis=1)

    for i in range(1, 6):
        completo[f'nota20_{i}'] = completo[f'nota20_{i}'].round(2)

    return completo


## 4. Ejecución del pipeline ETL
Se itera sobre cada archivo semestral, se aplican las dos funciones de
transformación y se guarda el resultado en `Cleaning_Data/`.

In [ ]:
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

for archivo in sorted(CARPETA_ENTRADA.glob('*.xlsx')):
    print(f'Procesando  →  {archivo.name} ...', end=' ')
    try:
        hoja_param  = limpiar_parametros_liquidacion(archivo)
        df_limpio   = cleaning_function(archivo, hoja_param)

        nombre_out  = archivo.stem + '_ETLclean.xlsx'
        ruta_out    = CARPETA_SALIDA / nombre_out
        df_limpio.to_excel(ruta_out, index=False)

        print(f'OK  ({len(df_limpio):,} registros)  →  {ruta_out.name}')
    except Exception as e:
        print(f'ERROR: {e}')


## 5. Verificación de salidas
Se listan los archivos generados y se muestra un resumen del primero
para confirmar que la estructura es correcta.

In [ ]:
archivos_out = sorted(CARPETA_SALIDA.glob('*_ETLclean.xlsx'))
print(f'Archivos generados: {len(archivos_out)}\n')
for f in archivos_out:
    df_check = pd.read_excel(f, nrows=0)
    print(f'  {f.name:35s}  columnas: {len(df_check.columns)}')

# Vista previa del primer archivo
print('\nVista previa del primer archivo procesado:')
pd.read_excel(archivos_out[0]).head(3)
